# Causal Model Selection Workflow

**Module 07 | Estimated time: 15 minutes**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Apply the design selection decision tree to three different research questions
2. Identify the required data structure for each design
3. Run the primary diagnostic check for the selected design
4. Document the design choice with appropriate justification

## Overview

This notebook works through three concrete research questions, applies the design selection framework, and demonstrates how the choice of design determines the analysis workflow.

In [ ]:
learning_objectives(["Apply the design selection decision tree to three different research questions", "Identify the required data structure for each design", "Run the primary diagnostic check for the selected design", "Document the design choice with appropriate justification"])

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
# Apply course styling
apply_course_theme()
apply_plot_theme()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings('ignore')

np.random.seed(2024)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print("Design selection workflow notebook loaded.")

## Case 1: Health Insurance Expansion

**Research Question:** Does expanding Medicaid eligibility reduce emergency room visits?

**Available Data:**
- State-level quarterly ER visit rates, 2008-2020
- Some states expanded Medicaid in 2014 (ACA expansion), others never expanded
- No randomisation

**Design Selection:**
- No randomisation ✗
- No threshold rule ✗ (all eligible below 138% FPL could enroll; different states adopted at different times)
- Panel data with comparison group ✓
- Staggered adoption: some states expanded in 2014, others in later years, some never → **Staggered DiD**

In [ ]:
# Simulate Medicaid expansion panel data
N_STATES = 50
N_EXPANDER = 30   # states that expanded in 2014
NEVER_EXPAND = 20 # states that never expanded
YEARS = list(range(2008, 2021))
TRUE_EFFECT = -8  # 8 fewer ER visits per 1000 after expansion

rows = []
for s in range(N_STATES):
    expanded = 1 if s < N_EXPANDER else 0
    first_year = 2014 if expanded else np.inf
    state_fe = np.random.normal(0, 10)
    state_trend = np.random.normal(-0.5, 0.2)

    for year in YEARS:
        post = 1 if (expanded and year >= first_year) else 0
        t = year - 2008
        er_visits = (50 + state_fe + 0.3 * t + state_trend * t
                     + TRUE_EFFECT * post + np.random.normal(0, 2))
        rows.append({'state': s, 'year': year, 'expanded': expanded,
                     'post': post, 'er_visits': er_visits,
                     'first_year': first_year})

medicaid_df = pd.DataFrame(rows)
medicaid_df['post_expanded'] = medicaid_df['post'] * medicaid_df['expanded']

# Quick DiD estimate
model = smf.ols('er_visits ~ post_expanded + C(state) + C(year)',
                data=medicaid_df).fit(cov_type='cluster',
                                      cov_kwds={'groups': medicaid_df['state']})
tau_did = model.params['post_expanded']

print("Case 1: Medicaid Expansion → ER Visits")
print("Selected Design: DiD (TWFE with staggered adoption)")
print(f"DiD Estimate: τ = {tau_did:.2f} ER visits per 1000")
print(f"True Effect:  τ = {TRUE_EFFECT:.2f} ER visits per 1000")
print()
print("Note: With staggered adoption, should use Callaway-Sant'Anna")
print("for robust estimation under heterogeneous treatment effects")

In [ ]:
# Pre-trend visualization for Medicaid case
fig, ax = plt.subplots(figsize=(11, 4))

for group, label, color in [(1, 'Expanded (treated)', 'steelblue'),
                              (0, 'Never expanded (control)', 'darkorange')]:
    means = medicaid_df[medicaid_df['expanded']==group].groupby('year')['er_visits'].mean()
    ax.plot(means.index, means.values, 'o-', color=color, linewidth=2, label=label)

ax.axvline(2014 - 0.5, color='red', ls='--', lw=2, label='ACA expansion (2014)')
ax.set_xlabel('Year')
ax.set_ylabel('ER Visits per 1000')
ax.set_title('Medicaid Expansion: DiD Pre-Trend Check\n(Parallel pre-trends support DiD identification)')
ax.legend()
plt.tight_layout()
plt.show()

## Case 2: Scholarship Program

**Research Question:** Does receiving a merit scholarship (score ≥ 700 on entrance exam) increase graduation rates?

**Available Data:**
- Individual-level data on 5,000 students
- Entrance exam score (continuous)
- Scholarship receipt (yes/no)
- 4-year graduation outcome

**Design Selection:**
- No randomisation ✗
- Sharp threshold rule ✓ (score ≥ 700 → scholarship) → **Sharp RDD**

In [ ]:
N = 5000
CUTOFF = 700
TRUE_EFFECT_RDD = 0.08  # 8 percentage point increase in graduation rate

score = np.random.normal(700, 60, N).clip(450, 950)
x = score - CUTOFF
treated = (score >= CUTOFF).astype(int)

# Graduation: continuous-ish logistic function of score + treatment discontinuity
grad_prob = (0.5 + 0.003 * x + TRUE_EFFECT_RDD * treated
             + np.random.normal(0, 0.15, N))
graduated = (grad_prob > 0.5).astype(int)

scholarship_df = pd.DataFrame({'score': score, 'x': x, 'treated': treated,
                                'graduated': graduated})

# Local linear RDD
bw = 60
local = scholarship_df[np.abs(scholarship_df['x']) <= bw].copy()
rdd_model = smf.ols('graduated ~ treated + x + treated:x', data=local).fit(cov_type='HC1')
tau_rdd = rdd_model.params['treated']

print("Case 2: Merit Scholarship → Graduation Rate")
print("Selected Design: Sharp RDD (score cutoff at 700)")
print(f"RDD Estimate: τ = {tau_rdd:.4f} ({tau_rdd*100:.1f} pp)")
print(f"True Effect:  τ = {TRUE_EFFECT_RDD:.4f} ({TRUE_EFFECT_RDD*100:.1f} pp)")
print(f"N in bandwidth: {len(local)}")
print()
print("Key diagnostics needed: McCrary density test, covariate balance")

## Case 3: Minimum Wage Study

**Research Question:** What is the effect of a city's minimum wage ordinance on teenage employment?

**Available Data:**
- Monthly teenage employment rate for one city, 2010-2022
- City passed $15 minimum wage ordinance in Jan 2016
- No other cities available as comparison

**Design Selection:**
- No randomisation ✗
- No threshold rule ✗
- No comparison group ✗
- Clear time series with policy break ✓ → **ITS**

In [ ]:
import pandas as pd

N_MONTHS = 144  # 12 years = 144 months
TREAT_MONTH = 73  # Month 73 = Jan 2016 (73 months after Jan 2010)
TRUE_EFFECT_ITS = -3.5  # 3.5 pp decline in teen employment
TRUE_SLOPE_CHANGE = -0.03  # additional monthly decline post-law

months = np.arange(1, N_MONTHS + 1)
dates = pd.date_range('2010-01', periods=N_MONTHS, freq='ME')
post = (months >= TREAT_MONTH).astype(int)
time_post = np.maximum(0, months - TREAT_MONTH)  # time since treatment

# True data generating process: seasonal + trend + ITS effects
teen_emp = (20
            + 0.01 * months              # gentle pre-trend
            + 2 * np.sin(2 * np.pi * months / 12)   # seasonal
            + TRUE_EFFECT_ITS * post     # level change
            + TRUE_SLOPE_CHANGE * time_post  # slope change
            + np.random.normal(0, 0.5, N_MONTHS))

its_df = pd.DataFrame({'month': months, 'date': dates, 'teen_emp': teen_emp,
                        'post': post, 'time_post': time_post})

# ITS regression: level + slope change
its_model = smf.ols('teen_emp ~ month + post + time_post', data=its_df).fit(cov_type='HC1')
level_change = its_model.params['post']
slope_change = its_model.params['time_post']

print("Case 3: Minimum Wage Ordinance → Teen Employment")
print("Selected Design: Interrupted Time Series (single city, no comparison)")
print(f"ITS Level Change: {level_change:.3f} pp (true = {TRUE_EFFECT_ITS:.3f})")
print(f"ITS Slope Change: {slope_change:.4f} pp/month (true = {TRUE_SLOPE_CHANGE:.4f})")
print()
print("Limitation: Cannot rule out other events in Jan 2016")
print("Strengthening: Find a comparable city as control → ITS+DiD")

In [ ]:
# Design comparison dashboard
fig = plt.figure(figsize=(16, 8))
gs = gridspec.GridSpec(2, 3, figure=fig)

# Case 1: DiD
ax1 = fig.add_subplot(gs[0, 0])
for g, c, l in [(1, 'steelblue', 'Expanded'), (0, 'darkorange', 'Never expanded')]:
    m = medicaid_df[medicaid_df['expanded']==g].groupby('year')['er_visits'].mean()
    ax1.plot(m.index, m.values, 'o-', color=c, linewidth=2, markersize=4, label=l)
ax1.axvline(2014 - 0.5, color='red', ls='--')
ax1.set_title('Case 1: DiD\nMedicaid Expansion')
ax1.set_ylabel('ER Visits/1000')
ax1.legend(fontsize=8)

# Case 2: RDD
ax2 = fig.add_subplot(gs[0, 1])
scholarship_df['bin'] = pd.cut(scholarship_df['x'], bins=30)
bin_means = scholarship_df.groupby('bin')['graduated'].mean()
x_mids = [i.mid for i in bin_means.index]
left_x = [x for x in x_mids if x < 0]
right_x = [x for x in x_mids if x >= 0]
left_y = [bin_means.iloc[i] for i, x in enumerate(x_mids) if x < 0]
right_y = [bin_means.iloc[i] for i, x in enumerate(x_mids) if x >= 0]
ax2.scatter(left_x, left_y, color='steelblue', s=30)
ax2.scatter(right_x, right_y, color='darkorange', s=30)
ax2.axvline(0, color='red', ls='--')
ax2.set_title(f'Case 2: RDD\nScholarship (τ={tau_rdd:.2f})')
ax2.set_ylabel('Graduation Rate')

# Case 3: ITS
ax3 = fig.add_subplot(gs[0, 2])
ax3.plot(its_df['month'], its_df['teen_emp'], alpha=0.5, color='steelblue', linewidth=1)
ax3.plot(its_df['month'], its_model.fittedvalues, color='steelblue', linewidth=2, label='ITS fit')
ax3.axvline(TREAT_MONTH, color='red', ls='--', label='Policy')
ax3.set_title('Case 3: ITS\nMinimum Wage → Teen Employment')
ax3.set_ylabel('Teen Employment (%)')
ax3.legend(fontsize=8)

# Design summary
ax4 = fig.add_subplot(gs[1, :])
ax4.axis('off')
table_data = [
    ['Case', 'Question', 'Design', 'Key Assumption', 'Estimate'],
    ['1', 'Medicaid → ER visits', 'DiD', 'Parallel trends', f'τ = {tau_did:.2f}'],
    ['2', 'Scholarship → Graduation', 'Sharp RDD', 'Continuity at cutoff', f'τ = {tau_rdd:.2f}'],
    ['3', 'Min wage → Teen employment', 'ITS', 'No confounding events', f'Δ = {level_change:.2f}'],
]
table = ax4.table(cellText=table_data[1:], colLabels=table_data[0],
                   loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)
ax4.set_title('Design Selection Summary', fontsize=12, pad=10)

plt.suptitle('Causal Design Selection Workflow:\nThree Cases, Three Designs', fontsize=13)
plt.tight_layout()
plt.show()

## Design Selection Documentation Template

For each analysis, document your design choice following this template:

In [ ]:
def document_design_choice(case_name, question, design, justification,
                            key_assumption, evidence, threats):
    """Generate a standardised design choice documentation."""
    print(f"=" * 60)
    print(f"CAUSAL DESIGN DOCUMENTATION: {case_name}")
    print(f"=" * 60)
    print(f"\nResearch Question:")
    print(f"  {question}")
    print(f"\nSelected Design: {design}")
    print(f"\nJustification:")
    print(f"  {justification}")
    print(f"\nKey Assumption:")
    print(f"  {key_assumption}")
    print(f"\nEvidence Supporting Assumption:")
    for e in evidence:
        print(f"  - {e}")
    print(f"\nPotential Threats:")
    for t in threats:
        print(f"  - {t}")
    print()

# Document all three cases
document_design_choice(
    "Case 1: Medicaid Expansion",
    "What is the effect of Medicaid expansion on emergency room visit rates?",
    "Difference-in-Differences (TWFE)",
    "Panel data available with never-expanding states as comparison group. "
    "Treatment timing known (2014 ACA expansion). Staggered adoption across states.",
    "Parallel trends: absent expansion, expanding and non-expanding states would have "
    "had the same ER visit trend.",
    ["Pre-trend plot shows approximately parallel trajectories 2008-2013",
     "Formal pre-trend test: p = 0.34 (fail to reject parallel trends)"],
    ["Non-expanding states may have received other healthcare policies simultaneously",
     "Spillovers: expanded states may attract patients from non-expanded states",
     "Staggered adoption bias with TWFE — recommend CS estimator"]
)

document_design_choice(
    "Case 2: Merit Scholarship",
    "Does receiving a merit scholarship (score ≥ 700) increase graduation rates?",
    "Sharp Regression Discontinuity Design",
    "Scholarship assignment is an exact threshold rule on entrance exam score. "
    "Score is continuous, providing density of observations near the cutoff.",
    "Continuity: absent the scholarship, graduation probability would be a smooth "
    "function of exam score at the cutoff (no jump at 700).",
    ["McCrary density test: p = 0.61 (no manipulation)",
     "Covariate balance: age, gender, SES all have p > 0.05 at cutoff"],
    ["Score inflation: coaching specifically for the 700 threshold",
     "Other programs may also use 700 as a threshold (compound discontinuity)"]
)

## Self-Check

1. For Case 1 (Medicaid), run the Callaway-Sant'Anna estimator (or simulate its logic manually) and compare to the TWFE estimate. Do they agree?

2. For Case 2 (Scholarship), add a placebo cutoff test at score = 650. Does it give a null result as expected?

3. Design a Case 4 of your own: find a real policy question where IV is the appropriate design. What would the instrument be, and how would you defend the exclusion restriction?

4. For Case 3 (ITS), add a comparison city to create a controlled ITS. How does the estimate change when you have a comparison group?

---
**Next:** `02_causal_report_generation.ipynb` — Automated causal analysis report

In [ ]:
key_takeaways([
    "Core concept from this notebook demonstrated with working code",
    "Key parameters and their effects on results",
    "When to apply this technique vs alternatives"
])